In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

repo_root = Path.cwd().resolve()
if not (repo_root / "backtesting" / "renquant_102").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

import common

# ── Strategy configuration ───────────────────────────────────────────────
STRATEGY_DIR = repo_root / "backtesting" / "renquant_102"
config = common.load_strategy_config(STRATEGY_DIR / "strategy_config.json")

WATCHLIST = config["watchlist"]
BENCHMARK = config.get("benchmark", "SPY")
MODEL_NAME = config["model_name"]
START = config["sample_start"]
END = config["sample_end"]
BACKTEST_START = config["backtest_start"]
BACKTEST_END = config["backtest_end"]
PROVIDER = config["data_src"]
MODEL_PARAMS = config["model_params"]
TRAINING_YEARS = config.get("training_years", 2)

# Volume scanner params
ZSCORE_LOOKBACK = config.get("volume_zscore_lookback", 15)
ZSCORE_THRESHOLD = config.get("volume_zscore_threshold", 2.0)

# Position management
MAX_POSITIONS = config.get("max_concurrent_positions", 3)
INITIAL_CASH = config["initial_cash"]
WASH_SALE_DAYS = config.get("wash_sale_days", 30)
MIN_HOLD_DAYS = config.get("min_hold_days", 20)
MAX_HOLD_DAYS = config.get("max_hold_days", 150)
pos = config.get("position_sizing", {})
MAX_POSITION_PCT = pos.get("max_position_pct", 0.33)
CASH_RESERVE_PCT = pos.get("cash_reserve_pct", 0.10)

# Full indicator spec for research
INDICATOR_SPEC = {
    "rsi": {"period": 14}, "macd": {"fast": 12, "slow": 26, "signal": 9},
    "cci": {"period": 20}, "bbp": {"period": 20}, "stochastic": {"window": 14, "smooth": 3},
    "adx": {"period": 14}, "atr": {"period": 14}, "obv": {"signal_period": 20},
    "williams_r": {"period": 14}, "ema": {"period": 50}, "momentum": {"period": 10},
}

FEATURE_COLUMNS = MODEL_PARAMS["feature_columns"]
RATIO_FEATURES = {"rsi", "adx"}
DIFF_FEATURES = {"macd_hist", "cci", "bbp", "williams_r", "obv_slope"}

print(f"Strategy:       {MODEL_NAME}")
print(f"Pipeline:       DETECT (z-score>{ZSCORE_THRESHOLD}) -> CONFIRM (4 approaches) -> EXECUTE")
print(f"Watchlist:      {WATCHLIST}")
print(f"Z-score:        lookback={ZSCORE_LOOKBACK} days, threshold={ZSCORE_THRESHOLD}")
print(f"Training:       {TRAINING_YEARS} years ({START} -> {END})")
print(f"Backtest:       {BACKTEST_START} -> {BACKTEST_END}")
print(f"Max positions:  {MAX_POSITIONS}")

In [ ]:
# ── Fetch OHLCV + indicators for all watchlist stocks + SPY ──────────────
dfs_raw = {}   # raw OHLCV (for volume scanner)
dfs_ind = {}   # with indicators

for symbol in WATCHLIST + [BENCHMARK]:
    df = common.fetch_ohlcv(symbol, start=START, end=END, provider=PROVIDER)
    dfs_raw[symbol] = df.copy()
    dfs_ind[symbol] = common.compute_indicators(df, INDICATOR_SPEC)
    print(f"  {symbol}: {len(df)} bars ({df.index[0].date()} -> {df.index[-1].date()})")

df_spy = dfs_ind[BENCHMARK]
print(f"\nFetched {len(WATCHLIST)} stocks + {BENCHMARK}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# STAGE 1: DETECT — Volume Z-Score Scanner
# ══════════════════════════════════════════════════════════════════════════
# For each stock, compute rolling volume z-score:
#   zscore = (today_vol - mean_N) / std_N
# A z-score > 2.0 means volume is 2 standard deviations above the recent norm.

def compute_volume_zscore(df, lookback=ZSCORE_LOOKBACK):
    vol = df["volume"].astype(float)
    roll_mean = vol.rolling(lookback).mean()
    roll_std = vol.rolling(lookback).std()
    return ((vol - roll_mean) / roll_std.replace(0, np.nan)).dropna()

# Compute z-scores for all stocks
zscore_matrix = pd.DataFrame()
spike_events = []

for symbol in WATCHLIST:
    zs = compute_volume_zscore(dfs_raw[symbol])
    zscore_matrix[symbol] = zs
    # Collect spike events above threshold
    spikes = zs[zs >= ZSCORE_THRESHOLD]
    for date, score in spikes.items():
        spike_events.append({"date": date, "symbol": symbol, "zscore": score})

spike_df = pd.DataFrame(spike_events).sort_values("zscore", ascending=False)
print(f"Total spike events (z-score >= {ZSCORE_THRESHOLD}): {len(spike_df)}")
print(f"Spikes per stock:")
print(spike_df.groupby("symbol").size().sort_values(ascending=False))

# ── Chart 1: Volume Z-Score Heatmap ─────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 10), gridspec_kw={"height_ratios": [3, 1]})

# Heatmap: clip z-scores for color range
heatmap_data = zscore_matrix.T.clip(-2, 6)
ax = axes[0]
sns.heatmap(heatmap_data, ax=ax, cmap="YlOrRd", vmin=0, vmax=5,
            cbar_kws={"label": "Volume Z-Score"},
            xticklabels=50, yticklabels=True)
ax.set_title(f"Volume Z-Score Heatmap (lookback={ZSCORE_LOOKBACK} days, threshold={ZSCORE_THRESHOLD})")
ax.set_xlabel("Date")
ax.set_ylabel("Stock")

# Timeline scatter: spike events
ax2 = axes[1]
if not spike_df.empty:
    colors = {s: plt.cm.tab10(i) for i, s in enumerate(WATCHLIST)}
    for symbol in WATCHLIST:
        sub = spike_df[spike_df["symbol"] == symbol]
        if not sub.empty:
            ax2.scatter(sub["date"], sub["zscore"], label=symbol,
                       color=colors[symbol], s=sub["zscore"] * 15, alpha=0.7)
    ax2.axhline(y=ZSCORE_THRESHOLD, color="red", linestyle="--", alpha=0.5, label=f"Threshold ({ZSCORE_THRESHOLD})")
    ax2.set_ylabel("Z-Score")
    ax2.set_title("Volume Spike Events")
    ax2.legend(loc="upper right", fontsize=7, ncol=5)
    ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Show top 20 spikes
print(f"\nTop 20 volume spikes:")
spike_df.head(20)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# STAGE 2: Build relative feature frames for each stock (2-year training)
# ══════════════════════════════════════════════════════════════════════════
# For each watchlist stock, compute relative features (stock vs SPY) over
# the full sample period. This data is used by all 4 confirmation approaches.

# Build per-stock z-score series (for gating signals later)
zscores = {}
for symbol in WATCHLIST:
    zscores[symbol] = compute_volume_zscore(dfs_raw[symbol])

# Build per-stock relative feature DataFrames
stock_features = {}  # {symbol: DataFrame with close, relative features, trend features}

for symbol in WATCHLIST:
    df_stock = dfs_ind[symbol]
    common_idx = df_stock.index.intersection(df_spy.index)
    df_s = df_stock.loc[common_idx]
    df_b = df_spy.loc[common_idx]

    df = pd.DataFrame(index=common_idx)
    df["close"] = df_s["close"]
    df["close_spy"] = df_b["close"]
    df["volume"] = df_s["volume"]

    # Relative indicator features (for Classification model)
    for col in FEATURE_COLUMNS:
        if col in RATIO_FEATURES:
            df[col] = df_s[col] / df_b[col].replace(0, np.nan)
        elif col in DIFF_FEATURES:
            df[col] = df_s[col] - df_b[col]

    # Trend-following features (for Dual Momentum, Breakout)
    df["trend"] = df_s["close"] / df_s["close"].ewm(span=50, adjust=False).mean()
    df["trend_long"] = df_s["close"] / df_s["close"].ewm(span=200, adjust=False).mean()
    rel_price = df_s["close"] / df_b["close"]
    df["rel_mom_20d"] = rel_price.pct_change(20)
    df["rel_mom_60d"] = rel_price.pct_change(60)

    # Breakout features
    df["high_20d_break"] = df_s["close"] - df_s["close"].rolling(20).max()
    df["low_20d_break"] = df_s["close"] - df_s["close"].rolling(20).min()

    # Volume z-score (align to feature index)
    if symbol in zscores:
        df["vol_zscore"] = zscores[symbol].reindex(common_idx)

    df = df.dropna()
    stock_features[symbol] = df
    print(f"  {symbol}: {len(df)} bars with features")

# ── Chart 2: Feature distribution summary ────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, col in enumerate(FEATURE_COLUMNS):
    ax = axes[i // 4][i % 4]
    for symbol in WATCHLIST[:5]:  # top 5 for readability
        if col in stock_features[symbol].columns:
            stock_features[symbol][col].hist(ax=ax, bins=30, alpha=0.4, label=symbol)
    ax.set_title(col, fontsize=10)
    ax.legend(fontsize=6)
# Hide unused subplot if odd number of features
for j in range(len(FEATURE_COLUMNS), 8):
    axes[j // 4][j % 4].set_visible(False)
plt.suptitle("Relative Feature Distributions (stock vs SPY)", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# SHARED: Constraint + multi-stock portfolio simulation
# ══════════════════════════════════════════════════════════════════════════

def apply_constraints(signals, min_hold_days=0, wash_sale_days=0, max_hold_days=0):
    """Filter raw signals to enforce holding period, max hold, and wash-sale rules."""
    out = signals.values.copy()
    dates = signals.index
    position = 0
    entry_idx = None
    last_sell_idx = None
    for i in range(len(out)):
        sig = out[i]
        if position == 1 and max_hold_days > 0 and entry_idx is not None:
            if (dates[i] - dates[entry_idx]).days >= max_hold_days:
                out[i] = "sell"
                position = 0
                last_sell_idx = i
                entry_idx = None
                continue
        if sig == "buy" and position == 0:
            if last_sell_idx is not None and (dates[i] - dates[last_sell_idx]).days < wash_sale_days:
                out[i] = "hold"
                continue
            position = 1
            entry_idx = i
        elif sig == "sell" and position == 1:
            if entry_idx is not None and (dates[i] - dates[entry_idx]).days < min_hold_days:
                out[i] = "hold"
                continue
            position = 0
            last_sell_idx = i
            entry_idx = None
        else:
            if (sig == "sell" and position == 0) or (sig == "buy" and position == 1):
                out[i] = "hold"
    return pd.Series(out, index=signals.index)


def gate_by_volume_spike(signals, vol_zscore, threshold=ZSCORE_THRESHOLD):
    """Only allow buy signals on days with volume z-score above threshold."""
    gated = signals.copy()
    zs_aligned = vol_zscore.reindex(signals.index).fillna(0)
    # Only gate BUY signals — sell signals should always be allowed
    mask = (gated == "buy") & (zs_aligned < threshold)
    gated[mask] = "hold"
    return gated


def simulate_multi_stock_equity(
    approach_signals,  # {symbol: pd.Series of constrained buy/sell/hold}
    approach_prices,   # {symbol: pd.Series of close prices}
    vol_zscores,       # {symbol: pd.Series of z-scores} for ranking
    max_positions=MAX_POSITIONS,
    initial_cash=INITIAL_CASH,
    max_position_pct=MAX_POSITION_PCT,
    cash_reserve_pct=CASH_RESERVE_PCT,
):
    """Simulate portfolio across multiple stocks with position limits."""
    # Build common date index
    all_dates = sorted(set().union(*[s.index for s in approach_signals.values()]))
    cash = float(initial_cash)
    holdings = {}  # {symbol: shares}
    equity = []
    trades = []

    for date in all_dates:
        # Current portfolio value
        port_value = cash + sum(
            holdings.get(s, 0) * float(approach_prices[s].get(date, approach_prices[s].iloc[-1]))
            for s in holdings if holdings[s] > 0
        )

        # Process SELLS first
        for symbol in list(holdings.keys()):
            if holdings[symbol] <= 0:
                continue
            sig = approach_signals.get(symbol, pd.Series(dtype=str))
            if date in sig.index and sig.loc[date] == "sell":
                price = float(approach_prices[symbol].loc[date])
                cash += holdings[symbol] * price
                trades.append({"date": date, "symbol": symbol, "action": "sell", "shares": holdings[symbol], "price": price})
                holdings[symbol] = 0

        # Count current positions
        current_positions = sum(1 for s in holdings if holdings.get(s, 0) > 0)
        open_slots = max_positions - current_positions

        if open_slots > 0:
            # Collect buy candidates with z-scores for ranking
            buy_candidates = []
            for symbol in approach_signals:
                if holdings.get(symbol, 0) > 0:
                    continue
                sig = approach_signals[symbol]
                if date in sig.index and sig.loc[date] == "buy":
                    zs = vol_zscores.get(symbol, pd.Series(dtype=float))
                    zscore_val = float(zs.get(date, 0)) if date in zs.index else 0
                    buy_candidates.append((symbol, zscore_val))

            # Rank by z-score descending
            buy_candidates.sort(key=lambda x: x[1], reverse=True)

            for symbol, zscore_val in buy_candidates[:open_slots]:
                if symbol not in approach_prices or date not in approach_prices[symbol].index:
                    continue
                price = float(approach_prices[symbol].loc[date])
                port_value_now = cash + sum(
                    holdings.get(s, 0) * float(approach_prices[s].get(date, 0))
                    for s in holdings if holdings.get(s, 0) > 0
                )
                reserve = port_value_now * cash_reserve_pct
                investable = max(cash - reserve, 0)
                max_invest = port_value_now * max_position_pct
                invest = min(max_invest, investable)
                shares = int(invest / price) if price > 0 else 0
                if shares > 0:
                    cash -= shares * price
                    holdings[symbol] = holdings.get(symbol, 0) + shares
                    trades.append({"date": date, "symbol": symbol, "action": "buy",
                                   "shares": shares, "price": price, "zscore": zscore_val})
                    open_slots -= 1
                    if open_slots <= 0:
                        break

        # Record equity
        total = cash + sum(
            holdings.get(s, 0) * float(approach_prices[s].get(date, approach_prices[s].iloc[-1]))
            for s in holdings if holdings.get(s, 0) > 0
        )
        equity.append({"date": date, "equity": total})

    eq_series = pd.Series(
        [e["equity"] for e in equity],
        index=pd.DatetimeIndex([e["date"] for e in equity])
    )
    return eq_series, trades


# ══════════════════════════════════════════════════════════════════════════
# APPROACH 1: Dual Momentum — Trend Following Confirmation
# ══════════════════════════════════════════════════════════════════════════
# Volume spike detected → check if stock is in uptrend + outperforming SPY.
# If yes, ride the momentum. Same rules as renquant_101 Manual model.

DUAL_MOM_RULES = [
    {"col": "trend",       "buy_above": 1.0,    "sell_below": 0.97},
    {"col": "trend_long",  "buy_above": 1.0,    "sell_below": 0.97},
    {"col": "rel_mom_20d", "buy_above": 0.0,    "sell_below": -0.03},
    {"col": "rel_mom_60d", "buy_above": 0.0,    "sell_below": -0.05},
    {"col": "macd_hist",   "buy_above": 0,       "sell_below": 0},
    {"col": "obv_slope",   "buy_above": 0,       "sell_below": 0},
]

dm_signals = {}
dm_prices = {}

for symbol in WATCHLIST:
    df = stock_features[symbol]
    model = common.create_model("manual", score_rules=DUAL_MOM_RULES,
                                 buy_threshold=4, sell_threshold=-3)
    model.train(df)
    raw = model.predict_bulk(df)
    gated = gate_by_volume_spike(raw, zscores[symbol])
    constrained = apply_constraints(gated, MIN_HOLD_DAYS, WASH_SALE_DAYS, MAX_HOLD_DAYS)
    dm_signals[symbol] = constrained
    dm_prices[symbol] = df["close"]

dm_equity, dm_trades = simulate_multi_stock_equity(dm_signals, dm_prices, zscores)

# ── Chart 3: Dual Momentum signals + equity ──────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
ax1, ax2 = axes

# Equity curve
norm_eq = dm_equity / dm_equity.iloc[0]
ax1.plot(norm_eq.index, norm_eq, label="Dual Momentum Portfolio", linewidth=1.5)
buy_trades = [t for t in dm_trades if t["action"] == "buy"]
sell_trades = [t for t in dm_trades if t["action"] == "sell"]
if buy_trades:
    ax1.scatter([t["date"] for t in buy_trades], norm_eq.reindex([t["date"] for t in buy_trades]),
               marker="^", color="green", s=60, zorder=5, label="Buy")
if sell_trades:
    ax1.scatter([t["date"] for t in sell_trades], norm_eq.reindex([t["date"] for t in sell_trades]),
               marker="v", color="red", s=60, zorder=5, label="Sell")
ax1.set_title(f"Approach 1: Dual Momentum — {len(buy_trades)} buys, {len(sell_trades)} sells")
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

# Trade distribution by stock
trade_stocks = [t["symbol"] for t in dm_trades if t["action"] == "buy"]
if trade_stocks:
    pd.Series(trade_stocks).value_counts().plot.bar(ax=ax2, color="steelblue")
    ax2.set_title("Buy Trades by Stock")
    ax2.set_ylabel("Count")

plt.tight_layout()
plt.show()

ret = dm_equity.iloc[-1] / dm_equity.iloc[0] - 1
daily = dm_equity.pct_change().dropna()
sharpe = daily.mean() / daily.std() * np.sqrt(252) if daily.std() > 0 else 0
print(f"Dual Momentum: Return={ret:.2%}, Sharpe={sharpe:.2f}, Trades={len(buy_trades)}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# APPROACH 2: Classification (Random Forest) — ML Confirmation
# ══════════════════════════════════════════════════════════════════════════
# Train a per-stock RF classifier on 2-year relative features.
# Volume spike gates buy signals; sell signals always pass through.

clf_signals = {}
clf_prices = {}
clf_meta = {}

for symbol in WATCHLIST:
    df = stock_features[symbol]
    model = common.create_model(
        "classification",
        feature_columns=FEATURE_COLUMNS,
        lookahead=MODEL_PARAMS["lookahead"],
        threshold=MODEL_PARAMS["threshold"],
        leaf_size=MODEL_PARAMS["leaf_size"],
        bags=MODEL_PARAMS["bags"],
        impact=MODEL_PARAMS.get("impact", 0.0),
        buy_threshold=MODEL_PARAMS.get("buy_threshold", 0.5),
        sell_threshold=MODEL_PARAMS.get("sell_threshold", -0.5),
    )
    meta = model.train(df)
    clf_meta[symbol] = meta
    raw = model.predict_bulk(df)
    gated = gate_by_volume_spike(raw, zscores[symbol])
    constrained = apply_constraints(gated, MIN_HOLD_DAYS, WASH_SALE_DAYS, MAX_HOLD_DAYS)
    clf_signals[symbol] = constrained
    clf_prices[symbol] = df["close"]
    dist = meta["label_distribution"]
    print(f"  {symbol}: {meta['train_rows']} rows, labels: +{dist['long']}/0:{dist['hold']}/-{dist['short']}")

clf_equity, clf_trades = simulate_multi_stock_equity(clf_signals, clf_prices, zscores)

# ── Chart 4: Classification signals + equity ─────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
ax1, ax2 = axes

norm_eq = clf_equity / clf_equity.iloc[0]
ax1.plot(norm_eq.index, norm_eq, label="Classification (RF) Portfolio", linewidth=1.5, color="darkorange")
buy_trades = [t for t in clf_trades if t["action"] == "buy"]
sell_trades = [t for t in clf_trades if t["action"] == "sell"]
if buy_trades:
    ax1.scatter([t["date"] for t in buy_trades], norm_eq.reindex([t["date"] for t in buy_trades]),
               marker="^", color="green", s=60, zorder=5, label="Buy")
if sell_trades:
    ax1.scatter([t["date"] for t in sell_trades], norm_eq.reindex([t["date"] for t in sell_trades]),
               marker="v", color="red", s=60, zorder=5, label="Sell")
ax1.set_title(f"Approach 2: Classification (RF) — {len(buy_trades)} buys, {len(sell_trades)} sells")
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

trade_stocks = [t["symbol"] for t in clf_trades if t["action"] == "buy"]
if trade_stocks:
    pd.Series(trade_stocks).value_counts().plot.bar(ax=ax2, color="darkorange")
    ax2.set_title("Buy Trades by Stock")
    ax2.set_ylabel("Count")

plt.tight_layout()
plt.show()

ret = clf_equity.iloc[-1] / clf_equity.iloc[0] - 1
daily = clf_equity.pct_change().dropna()
sharpe = daily.mean() / daily.std() * np.sqrt(252) if daily.std() > 0 else 0
print(f"Classification: Return={ret:.2%}, Sharpe={sharpe:.2f}, Trades={len(buy_trades)}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# APPROACH 3: Mean Reversion — Contrarian Confirmation
# ══════════════════════════════════════════════════════════════════════════
# Volume spike + oversold conditions = buy the dip.
# Contrarian counterpoint to Dual Momentum: buy when indicators show
# the stock is beaten down, expecting a reversion to the mean.
#
# Rules (score-based):
#   buy: rsi ratio < 0.9 (oversold vs SPY), bbp < -0.2, williams_r diff < -20,
#         trend < 0.98 (below 50EMA = dip), cci < -30
#   sell: rsi ratio > 1.1 (overbought), bbp > 0.3, trend > 1.05 (extended)

MR_RULES = [
    {"col": "rsi",        "buy_below": 0.9,    "sell_above": 1.1},     # ratio feature
    {"col": "bbp",        "buy_below": -0.2,   "sell_above": 0.3},     # diff feature
    {"col": "williams_r", "buy_below": -20,    "sell_above": 20},      # diff feature
    {"col": "trend",      "buy_below": 0.98,   "sell_above": 1.05},
    {"col": "cci",        "buy_below": -30,    "sell_above": 50},      # diff feature
]

def mean_reversion_score(row, rules):
    """Score: +1 for each oversold signal (buy), -1 for each overbought (sell)."""
    score = 0
    for r in rules:
        val = row.get(r["col"], np.nan)
        if pd.isna(val):
            continue
        if val < r["buy_below"]:
            score += 1
        elif val > r["sell_above"]:
            score -= 1
    return score

mr_signals = {}
mr_prices = {}

for symbol in WATCHLIST:
    df = stock_features[symbol]
    scores = df.apply(lambda row: mean_reversion_score(row, MR_RULES), axis=1)
    raw = pd.Series("hold", index=df.index)
    raw[scores >= 3] = "buy"      # 3 of 5 oversold indicators agree
    raw[scores <= -2] = "sell"    # 2 of 5 overbought indicators agree
    gated = gate_by_volume_spike(raw, zscores[symbol])
    constrained = apply_constraints(gated, MIN_HOLD_DAYS, WASH_SALE_DAYS, MAX_HOLD_DAYS)
    mr_signals[symbol] = constrained
    mr_prices[symbol] = df["close"]

mr_equity, mr_trades = simulate_multi_stock_equity(mr_signals, mr_prices, zscores)

# ── Chart 5: Mean Reversion signals + equity ─────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
ax1, ax2 = axes

norm_eq = mr_equity / mr_equity.iloc[0]
ax1.plot(norm_eq.index, norm_eq, label="Mean Reversion Portfolio", linewidth=1.5, color="purple")
buy_trades = [t for t in mr_trades if t["action"] == "buy"]
sell_trades = [t for t in mr_trades if t["action"] == "sell"]
if buy_trades:
    ax1.scatter([t["date"] for t in buy_trades], norm_eq.reindex([t["date"] for t in buy_trades]),
               marker="^", color="green", s=60, zorder=5, label="Buy")
if sell_trades:
    ax1.scatter([t["date"] for t in sell_trades], norm_eq.reindex([t["date"] for t in sell_trades]),
               marker="v", color="red", s=60, zorder=5, label="Sell")
ax1.set_title(f"Approach 3: Mean Reversion — {len(buy_trades)} buys, {len(sell_trades)} sells")
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

trade_stocks = [t["symbol"] for t in mr_trades if t["action"] == "buy"]
if trade_stocks:
    pd.Series(trade_stocks).value_counts().plot.bar(ax=ax2, color="purple")
    ax2.set_title("Buy Trades by Stock")
    ax2.set_ylabel("Count")

plt.tight_layout()
plt.show()

ret = mr_equity.iloc[-1] / mr_equity.iloc[0] - 1
daily = mr_equity.pct_change().dropna()
sharpe = daily.mean() / daily.std() * np.sqrt(252) if daily.std() > 0 else 0
print(f"Mean Reversion: Return={ret:.2%}, Sharpe={sharpe:.2f}, Trades={len(buy_trades)}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# APPROACH 4: Breakout — Momentum Confirmation
# ══════════════════════════════════════════════════════════════════════════
# Volume spike + price breaking above 20-day high + uptrend = ride the breakout.
# Pure momentum play: the volume confirms institutional interest.
#
# Rules (score-based):
#   buy: high_20d_break >= 0 (at/above 20d high), trend > 1.0, trend_long > 1.0,
#        rel_mom_20d > 0 (outperforming SPY), macd_hist > 0 (momentum positive)
#   sell: low_20d_break <= 0 (at/below 20d low), trend < 0.97, rel_mom_20d < -0.03

BO_RULES = [
    {"col": "high_20d_break", "buy_above": 0,     "sell_below": None},
    {"col": "trend",          "buy_above": 1.0,   "sell_below": 0.97},
    {"col": "trend_long",     "buy_above": 1.0,   "sell_below": 0.97},
    {"col": "rel_mom_20d",    "buy_above": 0.0,   "sell_below": -0.03},
    {"col": "macd_hist",      "buy_above": 0,     "sell_below": None},
    {"col": "low_20d_break",  "buy_above": None,  "sell_below": 0},
]

def breakout_score(row, rules):
    """Score: +1 for each breakout buy signal, -1 for each breakdown sell signal."""
    score = 0
    for r in rules:
        val = row.get(r["col"], np.nan)
        if pd.isna(val):
            continue
        if r["buy_above"] is not None and val > r["buy_above"]:
            score += 1
        if r["sell_below"] is not None and val < r["sell_below"]:
            score -= 1
    return score

bo_signals = {}
bo_prices = {}

for symbol in WATCHLIST:
    df = stock_features[symbol]
    scores = df.apply(lambda row: breakout_score(row, BO_RULES), axis=1)
    raw = pd.Series("hold", index=df.index)
    raw[scores >= 4] = "buy"      # 4 of 6 breakout indicators (strong confirmation)
    raw[scores <= -2] = "sell"    # 2 of 6 breakdown indicators
    gated = gate_by_volume_spike(raw, zscores[symbol])
    constrained = apply_constraints(gated, MIN_HOLD_DAYS, WASH_SALE_DAYS, MAX_HOLD_DAYS)
    bo_signals[symbol] = constrained
    bo_prices[symbol] = df["close"]

bo_equity, bo_trades = simulate_multi_stock_equity(bo_signals, bo_prices, zscores)

# ── Chart 6: Breakout signals + equity ───────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
ax1, ax2 = axes

norm_eq = bo_equity / bo_equity.iloc[0]
ax1.plot(norm_eq.index, norm_eq, label="Breakout Portfolio", linewidth=1.5, color="teal")
buy_trades = [t for t in bo_trades if t["action"] == "buy"]
sell_trades = [t for t in bo_trades if t["action"] == "sell"]
if buy_trades:
    ax1.scatter([t["date"] for t in buy_trades], norm_eq.reindex([t["date"] for t in buy_trades]),
               marker="^", color="green", s=60, zorder=5, label="Buy")
if sell_trades:
    ax1.scatter([t["date"] for t in sell_trades], norm_eq.reindex([t["date"] for t in sell_trades]),
               marker="v", color="red", s=60, zorder=5, label="Sell")
ax1.set_title(f"Approach 4: Breakout — {len(buy_trades)} buys, {len(sell_trades)} sells")
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

trade_stocks = [t["symbol"] for t in bo_trades if t["action"] == "buy"]
if trade_stocks:
    pd.Series(trade_stocks).value_counts().plot.bar(ax=ax2, color="teal")
    ax2.set_title("Buy Trades by Stock")
    ax2.set_ylabel("Count")

plt.tight_layout()
plt.show()

ret = bo_equity.iloc[-1] / bo_equity.iloc[0] - 1
daily = bo_equity.pct_change().dropna()
sharpe = daily.mean() / daily.std() * np.sqrt(252) if daily.std() > 0 else 0
print(f"Breakout: Return={ret:.2%}, Sharpe={sharpe:.2f}, Trades={len(buy_trades)}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# COMPARE: All 4 Approaches + SPY Benchmark
# ══════════════════════════════════════════════════════════════════════════

approaches = {
    "Dual Momentum": {"equity": dm_equity, "trades": dm_trades, "color": "steelblue"},
    "Classification": {"equity": clf_equity, "trades": clf_trades, "color": "darkorange"},
    "Mean Reversion": {"equity": mr_equity, "trades": mr_trades, "color": "purple"},
    "Breakout":       {"equity": bo_equity, "trades": bo_trades, "color": "teal"},
}

# SPY buy-and-hold benchmark (same period)
common_start = max(eq["equity"].index[0] for eq in approaches.values())
common_end = min(eq["equity"].index[-1] for eq in approaches.values())
spy_close = dfs_ind[BENCHMARK]["close"].loc[common_start:common_end]
spy_norm = spy_close / spy_close.iloc[0]

# ── Chart 7: Combined equity curves ─────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 10), gridspec_kw={"height_ratios": [3, 1]})
ax1, ax2 = axes

ax1.plot(spy_norm.index, spy_norm, label="SPY (Buy & Hold)", color="gray",
         linewidth=1.5, linestyle="--", alpha=0.7)

for name, data in approaches.items():
    eq = data["equity"].loc[common_start:common_end]
    norm = eq / eq.iloc[0]
    ax1.plot(norm.index, norm, label=name, color=data["color"], linewidth=1.5)

ax1.set_title("renquant-102: Volume Z-Score Scanner — 4 Approaches vs SPY")
ax1.set_ylabel("Normalized Equity")
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# Drawdown comparison
for name, data in approaches.items():
    eq = data["equity"].loc[common_start:common_end]
    running_max = eq.cummax()
    dd = (eq - running_max) / running_max
    ax2.plot(dd.index, dd, label=name, color=data["color"], linewidth=1, alpha=0.7)

ax2.set_title("Drawdown")
ax2.set_ylabel("Drawdown %")
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ── Summary table ────────────────────────────────────────────────────────
rows = []
for name, data in approaches.items():
    eq = data["equity"].loc[common_start:common_end]
    ret = eq.iloc[-1] / eq.iloc[0] - 1
    daily = eq.pct_change().dropna()
    sharpe = daily.mean() / daily.std() * np.sqrt(252) if daily.std() > 0 else 0
    running_max = eq.cummax()
    max_dd = ((eq - running_max) / running_max).min()
    n_buys = sum(1 for t in data["trades"] if t["action"] == "buy")
    n_sells = sum(1 for t in data["trades"] if t["action"] == "sell")
    rows.append({
        "Approach": name,
        "Return": f"{ret:.2%}",
        "Sharpe": f"{sharpe:.2f}",
        "Max Drawdown": f"{max_dd:.2%}",
        "Buys": n_buys,
        "Sells": n_sells,
    })

# Add SPY benchmark
spy_ret = spy_norm.iloc[-1] - 1
spy_daily = spy_close.pct_change().dropna()
spy_sharpe = spy_daily.mean() / spy_daily.std() * np.sqrt(252) if spy_daily.std() > 0 else 0
spy_max_dd = ((spy_close - spy_close.cummax()) / spy_close.cummax()).min()
rows.append({
    "Approach": "SPY (Benchmark)",
    "Return": f"{spy_ret:.2%}",
    "Sharpe": f"{spy_sharpe:.2f}",
    "Max Drawdown": f"{spy_max_dd:.2%}",
    "Buys": "-",
    "Sells": "-",
})

summary = pd.DataFrame(rows)
print("\n" + "=" * 70)
print("COMPARISON SUMMARY")
print("=" * 70)
print(summary.to_string(index=False))
print("=" * 70)

# Identify best approach by Sharpe
best_name = max(
    [(name, float(data["equity"].loc[common_start:common_end].pct_change().dropna().mean()
                   / data["equity"].loc[common_start:common_end].pct_change().dropna().std() * np.sqrt(252)))
     for name, data in approaches.items()],
    key=lambda x: x[1]
)
print(f"\nBest approach by Sharpe: {best_name[0]} ({best_name[1]:.2f})")
BEST_APPROACH = best_name[0]

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# EXPORT: Save best approach's per-stock model artifacts
# ══════════════════════════════════════════════════════════════════════════

export_dir = STRATEGY_DIR
print(f"Best approach: {BEST_APPROACH}")
print(f"Exporting to: {export_dir}")
print()

if BEST_APPROACH == "Classification":
    # Re-train and save per-stock RF models
    for symbol in WATCHLIST:
        df = stock_features[symbol]
        model = common.create_model(
            "classification",
            feature_columns=FEATURE_COLUMNS,
            lookahead=MODEL_PARAMS["lookahead"],
            threshold=MODEL_PARAMS["threshold"],
            leaf_size=MODEL_PARAMS["leaf_size"],
            bags=MODEL_PARAMS["bags"],
            impact=MODEL_PARAMS.get("impact", 0.0),
            buy_threshold=MODEL_PARAMS.get("buy_threshold", 0.5),
            sell_threshold=MODEL_PARAMS.get("sell_threshold", -0.5),
        )
        model.train(df)
        artifact_name = f"{MODEL_NAME}-{symbol}"
        meta = model.save(export_dir, artifact_name)
        print(f"  {symbol}: saved {artifact_name}-rf-trees.json + policy-metadata.json")

elif BEST_APPROACH == "Dual Momentum":
    # Save manual model with dual momentum rules per stock
    for symbol in WATCHLIST:
        model = common.create_model("manual", score_rules=DUAL_MOM_RULES,
                                     buy_threshold=4, sell_threshold=-3)
        artifact_name = f"{MODEL_NAME}-{symbol}"
        meta = model.save(export_dir, artifact_name)
        print(f"  {symbol}: saved {artifact_name}-policy-metadata.json")

elif BEST_APPROACH == "Mean Reversion":
    # Export mean reversion as manual model with MR rules
    mr_score_rules = [
        {"col": r["col"],
         "buy_above": None, "sell_below": r["buy_below"],   # inverted: buy when LOW
         "buy_below": None, "sell_above": r["sell_above"]}
        for r in MR_RULES
    ]
    # Use ManualModel's save with custom score_rules
    import json
    for symbol in WATCHLIST:
        artifact_name = f"{MODEL_NAME}-{symbol}"
        meta = {
            "model_name": artifact_name,
            "policy_type": "manual",
            "approach": "mean_reversion",
            "rules": MR_RULES,
            "buy_threshold": 3,
            "sell_threshold": -2,
        }
        meta_path = export_dir / f"{artifact_name}-policy-metadata.json"
        meta_path.write_text(json.dumps(meta, indent=2))
        print(f"  {symbol}: saved {artifact_name}-policy-metadata.json")

elif BEST_APPROACH == "Breakout":
    import json
    for symbol in WATCHLIST:
        artifact_name = f"{MODEL_NAME}-{symbol}"
        meta = {
            "model_name": artifact_name,
            "policy_type": "manual",
            "approach": "breakout",
            "rules": BO_RULES,
            "buy_threshold": 4,
            "sell_threshold": -2,
        }
        meta_path = export_dir / f"{artifact_name}-policy-metadata.json"
        meta_path.write_text(json.dumps(meta, indent=2))
        print(f"  {symbol}: saved {artifact_name}-policy-metadata.json")

print(f"\nExported {len(WATCHLIST)} per-stock artifacts for '{BEST_APPROACH}' approach.")
print("Ready for LEAN backtest: cd backtesting/renquant_102 && lean backtest .")